# Descenso por gradiente: como ajustan los pesos

**Objetivo de aprendizaje:**
Al finalizar este notebook, el alumno podrá:
- Calcular un gradiente numérico y entender que significa matemáticamente
- Visualizar la superficie de pérdida y la trayectoria del descenso
- Comparar el efecto de distintos learning rates en la convergencia
- Aplicar manualmente la regla de actualización `w = w - lr * dw`
- Decidir cuando NO usar una red neuronal usando el criterio de no-uso

**Conexión con el documento:**
Este notebook acompaña la sección `### 3.2 Descenso por gradiente` de `estructura_contenidos.md`.
Los conceptos de función de pérdida y curva de pérdida se trabajaron en B02B.

> Antes de seguir: si tuviera que explicarle a alguien sin formación técnica
> que hace el algoritmo de entrenamiento de una red neuronal,
> ¿con que analogía lo describiría?

<details>
<summary>Orientación para el instructor (desplegar tras la reflexión)</summary>

Una respuesta madura menciona: la idea de bajar una colina, que el algoritmo
usa la pendiente del terreno para decidir la dirección, y que el tamaño del paso
es una decisión que afecta tanto a la velocidad como a la estabilidad.

Si nadie responde, preguntar: 'Si estuvieras en la ladera de una montaña con
niebla y solo pudieras sentir la inclinación del suelo, como bajarías?'

Señal de comprensión: el alumno conecta la pendiente del terreno con la derivada
de la función de pérdida, y el tamaño del paso con el learning rate.

</details>

## Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
IMG = 'images'
os.makedirs(IMG, exist_ok=True)

import plotly
print('Librerias OK')
print(f'numpy: {np.__version__}  plotly: {plotly.__version__}')

Librerias OK
numpy: 1.26.4  plotly: 5.19.0


## 1. Ejemplo mínimo: el gradiente en la práctica

Función de pérdida simplificada con un solo peso: `L(w) = (w - 2)^2`
El mínimo es en `w = 2` donde `L = 0`.
Calculamos el gradiente en dos puntos para entender la dirección del descenso.

In [2]:
# Funcion de perdida: L(w) = (w - 2)^2
def L(w):
    return (w - 2.0)**2

def dL_dw(w):
    return 2.0 * (w - 2.0)  # derivada analitica

lr = 0.5

for w_pt, etiqueta in [(0.0, 'w = 0.0 (izquierda del minimo)'),
                        (4.0, 'w = 4.0 (derecha del minimo)')]:
    g = dL_dw(w_pt)
    w_nuevo = w_pt - lr * g
    print(f'{etiqueta}')
    print(f'  L(w)     = {L(w_pt):.2f}')
    print(f'  dL/dw    = {g:.2f}  ({"negativo: subir w reduce L" if g < 0 else "positivo: bajar w reduce L"})')
    print(f'  w_nuevo  = {w_pt} - {lr} * ({g:.1f}) = {w_nuevo:.2f}')
    # Verificacion con gradiente numerico
    delta = 0.001
    g_num = (L(w_pt + delta) - L(w_pt)) / delta
    print(f'  Gradiente numerico (aprox): {g_num:.4f}  <-> analitico: {g:.4f}')
    print()

w = 0.0 (izquierda del minimo)
  L(w)     = 4.00
  dL/dw    = -4.00  (negativo: subir w reduce L)
  w_nuevo  = 0.0 - 0.5 * (-4.0) = 2.00
  Gradiente numerico (aprox): -3.9990  <-> analitico: -4.0000

w = 4.0 (derecha del minimo)
  L(w)     = 4.00
  dL/dw    = 4.00  (positivo: bajar w reduce L)
  w_nuevo  = 4.0 - 0.5 * (4.0) = 2.00
  Gradiente numerico (aprox): 4.0010  <-> analitico: 4.0000



In [3]:
# Visualizacion: L(w) con gradiente en dos puntos
w_vals = np.linspace(-0.5, 5.0, 300)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(w_vals, L(w_vals), 'b-', lw=2.5, label='L(w) = (w-2)^2')
ax.axvline(x=2, color='green', linestyle='--', lw=1.5, label='Minimo (w=2, L=0)')

colores = [('#E65100', 0.0, 'w=0, grad=-4 (sube)'),
           ('#7B1FA2', 4.0, 'w=4, grad=+4 (baja)')]

for color, w_pt, lbl in colores:
    g = dL_dw(w_pt)
    Lpt = L(w_pt)
    # Tangente corta
    w_t = np.linspace(w_pt - 0.8, w_pt + 0.8, 30)
    ax.plot(w_t, Lpt + g * (w_t - w_pt), '--', color=color, lw=1.5, alpha=0.8)
    ax.plot(w_pt, Lpt, 'o', color=color, ms=11, label=lbl, zorder=5)
    # Flecha indicando direccion de actualizacion
    ax.annotate('', xy=(w_pt - 0.7 * np.sign(g), Lpt + 1.5),
                xytext=(w_pt, Lpt + 1.5),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))

ax.set_xlabel('Peso w', fontsize=12)
ax.set_ylabel('Perdida L(w)', fontsize=12)
ax.set_title('Gradiente en dos puntos: la flecha indica la direccion de actualizacion', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(-1, 10)
plt.tight_layout()
plt.savefig(f'{IMG}/B03E_fig01.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B03E_fig01.png')

Guardado: B03E_fig01.png


## 2. Ejemplo realista: descenso por gradiente en 2D

Dos pesos `w1` y `w2`. Función de pérdida:
`L(w1, w2) = (w1 - 1)^2 + 2*(w2 - 1)^2`

El mínimo global es en `(w1=1, w2=1)` donde `L=0`.
El eje `w2` tiene mayor curvatura (factor 2): el descenso es más rápido en `w2` que en `w1`,
lo que hace que la trayectoria no sea una línea recta al mínimo.

In [4]:
def L2(w1, w2):
    return (w1 - 1.0)**2 + 2.0*(w2 - 1.0)**2

def grad_L2(w1, w2):
    return np.array([2.0*(w1 - 1.0), 4.0*(w2 - 1.0)])

# Descenso desde (3, 3) con lr=0.2
lr_2d = 0.2
w = np.array([3.0, 3.0])
history = [w.copy()]

for step in range(30):
    g = grad_L2(w[0], w[1])
    w = w - lr_2d * g
    history.append(w.copy())

history = np.array(history)
print('Evolucion del descenso:')
for idx in [0, 3, 8, 15, 29]:
    print(f'  Paso {idx:2d}: w=({history[idx,0]:.3f}, {history[idx,1]:.3f})  L={L2(*history[idx]):.5f}')

Evolucion del descenso:
  Paso  0: w=(3.000, 3.000)  L=12.00000
  Paso  3: w=(1.432, 1.016)  L=0.18714
  Paso  8: w=(1.034, 1.000)  L=0.00113
  Paso 15: w=(1.001, 1.000)  L=0.00000
  Paso 29: w=(1.000, 1.000)  L=0.00000


In [5]:
# Curvas de nivel + trayectoria
w1v = np.linspace(-0.2, 3.5, 200)
w2v = np.linspace(-0.2, 3.5, 200)
W1g, W2g = np.meshgrid(w1v, w2v)
Zg = L2(W1g, W2g)

fig, ax = plt.subplots(figsize=(8, 7))
cf = ax.contourf(W1g, W2g, Zg, levels=25, cmap='RdYlGn_r', alpha=0.85)
ax.contour(W1g, W2g, Zg, levels=15, colors='white', alpha=0.25, linewidths=0.5)
plt.colorbar(cf, ax=ax, label='Perdida L(w1, w2)')

ax.plot(history[:, 0], history[:, 1], 'o-', color='#1565C0',
        lw=2, ms=4, label=f'Trayectoria (lr={lr_2d})', zorder=5)
ax.plot(history[0, 0], history[0, 1], 's', color='red', ms=13,
        label='Inicio (3,3)', zorder=6)
ax.plot(1, 1, '*', color='lime', ms=17, markeredgecolor='black',
        label='Minimo (1,1)', zorder=6)

for i in range(0, len(history)-1, 5):
    ax.annotate('', xy=history[i+1], xytext=history[i],
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.8))

ax.set_xlabel('Peso w1', fontsize=12)
ax.set_ylabel('Peso w2', fontsize=12)
ax.set_title('Descenso por gradiente 2D - curvas de nivel y trayectoria', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{IMG}/B03E_fig02.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B03E_fig02.png')

Guardado: B03E_fig02.png


## 3. Ejemplo avanzado: comparativa de learning rates

Misma función de pérdida `L(w) = (w - 3)^2`, misma posición inicial `w = 0`.
Tres learning rates muestran los tres regímenes de convergencia.

In [6]:
def L1(w): return (w - 3.0)**2
def dL1(w): return 2.0*(w - 3.0)

configs = [
    ('lr=0.05 (demasiado bajo)',  0.05, '#E65100'),
    ('lr=0.50 (optimo)',          0.50, '#2E7D32'),
    ('lr=1.10 (demasiado alto)',  1.10, '#B71C1C'),
]

n_steps = 40
results = {}
for label, lr_val, color in configs:
    w = 0.0
    ws, ls = [w], [L1(w)]
    for _ in range(n_steps):
        w = w - lr_val * dL1(w)
        if abs(w) > 200: break
        ws.append(w); ls.append(L1(w))
    results[label] = (np.array(ws), np.array(ls))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: trayectoria sobre L(w)
w_plot = np.linspace(-4, 8, 300)
ax = axes[0]
ax.plot(w_plot, L1(w_plot), 'k-', lw=2, label='L(w) = (w-3)^2')
ax.axvline(x=3, color='gray', linestyle='--', lw=1)
for label, lr_val, color in configs:
    ws, _ = results[label]
    shown = ws[:min(20, len(ws))]
    ax.plot(shown, L1(shown), 'o-', color=color, ms=4, lw=1.5,
            label=label, alpha=0.85)
ax.set_xlabel('Peso w', fontsize=11)
ax.set_ylabel('Perdida L(w)', fontsize=11)
ax.set_title('Trayectoria en la funcion de perdida', fontsize=12)
ax.set_ylim(-1, 35)
ax.set_xlim(-4, 8)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Derecha: curva de perdida por epoca
ax = axes[1]
for label, lr_val, color in configs:
    ws, ls = results[label]
    ax.plot(range(len(ls)), np.clip(ls, 0, 50), 'o-', color=color,
            ms=4, lw=2, label=label, alpha=0.9)
ax.set_xlabel('Epoca (paso)', fontsize=11)
ax.set_ylabel('Perdida L', fontsize=11)
ax.set_title('Curva de perdida segun el learning rate', fontsize=12)
ax.set_ylim(-1, 55)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Anotaciones de diagnostico
axes[1].annotate('Convergencia lenta\n(LR bajo)', xy=(35, 3.5),
                 color='#E65100', fontsize=8, ha='center')
axes[1].annotate('Divergencia / oscilacion\n(LR alto)', xy=(8, 48),
                 color='#B71C1C', fontsize=8, ha='center')
axes[1].annotate('Convergencia optima', xy=(12, 0.2),
                 color='#2E7D32', fontsize=8, ha='center')

plt.suptitle('Comparativa de learning rates - L(w) = (w-3)^2', fontsize=13)
plt.tight_layout()
plt.savefig(f'{IMG}/B03E_fig03.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B03E_fig03.png')

print()
print('Resumen de convergencia:')
for label, _, _ in configs:
    ws, ls = results[label]
    print(f'  {label}: L final = {ls[-1]:.4f} en {len(ls)} pasos')

Guardado: B03E_fig03.png

Resumen de convergencia:
  lr=0.05 (demasiado bajo): L final = 0.0020 en 41 pasos
  lr=0.50 (optimo): L final = 0.0000 en 41 pasos
  lr=1.10 (demasiado alto): L final = 27429.4649 en 23 pasos


## 4. Visualización 3D interactiva: superficie de pérdida

Paraboloide `L(w1, w2) = w1^2 + w2^2` con la trayectoria del descenso
desde el punto `(2.5, 2.5)` hasta el mínimo `(0, 0)`.

In [7]:
# Superficie 3D
w1r = np.linspace(-3, 3, 60)
w2r = np.linspace(-3, 3, 60)
W1_3d, W2_3d = np.meshgrid(w1r, w2r)
Z_3d = W1_3d**2 + W2_3d**2

# Descenso desde (2.5, 2.5)
w_gd = np.array([2.5, 2.5])
path = [w_gd.copy()]
for _ in range(25):
    g = 2.0 * w_gd
    w_gd = w_gd - 0.15 * g
    path.append(w_gd.copy())
path = np.array(path)
path_z = path[:,0]**2 + path[:,1]**2 + 0.15

fig3d = go.Figure()
fig3d.add_trace(go.Surface(
    x=w1r, y=w2r, z=Z_3d,
    colorscale='RdYlGn_r', opacity=0.82,
    showscale=True,
    colorbar=dict(title='Perdida L'),
    name='Superficie'
))
fig3d.add_trace(go.Scatter3d(
    x=path[:,0], y=path[:,1], z=path_z,
    mode='lines+markers',
    line=dict(color='#1565C0', width=6),
    marker=dict(size=5, color='#1565C0'),
    name='Trayectoria descenso'
))
fig3d.add_trace(go.Scatter3d(
    x=[path[0,0]], y=[path[0,1]], z=[path_z[0]],
    mode='markers',
    marker=dict(size=13, color='red', symbol='square'),
    name='Inicio (2.5, 2.5)'
))
fig3d.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0.15],
    mode='markers',
    marker=dict(size=13, color='lime', symbol='diamond'),
    name='Minimo (0, 0)'
))
fig3d.update_layout(
    title='Superficie de perdida 3D y trayectoria del descenso por gradiente',
    scene=dict(
        xaxis_title='Peso w1',
        yaxis_title='Peso w2',
        zaxis_title='Perdida L',
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.1))
    ),
    height=550,
    template='plotly_white'
)

html_path = f'{IMG}/B03E_fig04.html'
fig3d.write_html(html_path)
print(f'Guardado HTML: {html_path}')
try:
    fig3d.write_image(f'{IMG}/B03E_fig04.png', width=900, height=550)
    print('Guardado PNG: B03E_fig04.png')
except Exception as e:
    print(f'PNG no disponible (kaleido): {e}')
fig3d.show()

Guardado HTML: images/B03E_fig04.html
PNG no disponible (kaleido): 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 5. Variantes del descenso: mini-batch vs SGD vs full-batch

In [8]:
data = {
    'Variante':       ['Full-batch GD',   'SGD (1 muestra)', 'Mini-batch GD'],
    'Datos / paso':   ['Todo el dataset', '1 muestra',       '32-256 muestras'],
    'Gradiente':      ['Exacto',          'Muy ruidoso',     'Estable (estimacion)'],
    'Velocidad':      ['Lenta',           'Maxima',          'Optima (GPU)'],
    'GPU-eficiente':  ['No',              'No',              'Si'],
    'Uso tipico':     ['Modelos pequeños', 'Raro hoy',       'Estandar actual'],
}
df_b = pd.DataFrame(data)
print(df_b.to_string(index=False))

fig, ax = plt.subplots(figsize=(13, 2.8))
ax.axis('off')
t = ax.table(cellText=df_b.values, colLabels=df_b.columns,
             cellLoc='center', loc='center')
t.auto_set_font_size(False)
t.set_fontsize(9.5)
t.scale(1.1, 1.7)
for j in range(len(df_b.columns)):
    t[0, j].set_facecolor('#1565C0')
    t[0, j].set_text_props(color='white', fontweight='bold')
for j in range(len(df_b.columns)):
    t[1, j].set_facecolor('#ECEFF1')  # full-batch: gris
for j in range(len(df_b.columns)):
    t[2, j].set_facecolor('#FFF9C4')  # SGD: amarillo
for j in range(len(df_b.columns)):
    t[3, j].set_facecolor('#C8E6C9')  # mini-batch: verde (estandar)
plt.title('Variantes del descenso por gradiente (verde = estandar actual)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{IMG}/B03E_fig05.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B03E_fig05.png')

       Variante    Datos / paso            Gradiente    Velocidad GPU-eficiente       Uso tipico
  Full-batch GD Todo el dataset               Exacto        Lenta            No Modelos pequeños
SGD (1 muestra)       1 muestra          Muy ruidoso       Maxima            No         Raro hoy
  Mini-batch GD 32-256 muestras Estable (estimacion) Optima (GPU)            Si  Estandar actual
Guardado: B03E_fig05.png


In [9]:
# Esquema: ciclo forward-loss-backprop-actualizacion
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 3.5)
ax.axis('off')

bloques = [
    (1.3, 1.75, '#E3F2FD', '#1565C0', 'Forward pass',     'Red calcula prediccion'),
    (4.0, 1.75, '#FCE4EC', '#C62828', 'Funcion de perdida','Mide el error L'),
    (6.8, 1.75, '#E8F5E9', '#2E7D32', 'Backpropagation',   'Calcula dL/dW'),
    (9.6, 1.75, '#FFF8E1', '#F57F17', 'Actualizacion',     'w -= lr * dL/dW'),
]

for (x, y, cf, ct, titulo, desc) in bloques:
    rect = mpatches.FancyBboxPatch((x-1.1, y-0.65), 2.2, 1.3,
                                   boxstyle='round,pad=0.1', lw=1.5,
                                   edgecolor=ct, facecolor=cf)
    ax.add_patch(rect)
    ax.text(x, y+0.32, titulo, ha='center', va='center',
            fontsize=9, fontweight='bold', color=ct)
    ax.text(x, y-0.22, desc, ha='center', va='center',
            fontsize=8, color='#444444')

for x_from, x_to in [(2.42, 2.88), (5.12, 5.68), (7.92, 8.48)]:
    ax.annotate('', xy=(x_to, 1.75), xytext=(x_from, 1.75),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))

# Flecha retorno
ax.annotate('', xy=(1.3, 0.8), xytext=(9.6, 0.8),
            arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
ax.text(5.45, 0.5, 'proxima iteracion (siguiente batch)', ha='center',
        fontsize=8, color='#555555', style='italic')

ax.set_title('Ciclo de entrenamiento: forward - perdida - backprop - actualizacion',
             fontsize=11, pad=8)
plt.tight_layout()
plt.savefig(f'{IMG}/B03E_fig06.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B03E_fig06.png')

Guardado: B03E_fig06.png


## 6. Ejercicio técnico: actualización manual de pesos

**Enunciado:**
Una red con dos pesos `w1 = 0.5` y `w2 = -0.3` produce una pérdida de `L = 4.2`.
Los gradientes calculados por backpropagation son:
- `dL/dw1 = 2.0`
- `dL/dw2 = -1.5`

Con `learning_rate = 0.1`, calcula:
1. Los valores nuevos de `w1` y `w2` después de un paso de descenso.
2. ¿En que dirección se movió cada peso y por que?
3. Si `lr = 0.8`, ¿que cambia? ¿es un lr razonable?

In [10]:
# Datos del ejercicio - no modificar
w1 = 0.5
w2 = -0.3
L_ini = 4.2
dw1 = 2.0
dw2 = -1.5
lr  = 0.1

print('Estado inicial:')
print(f'  w1={w1}  dL/dw1={dw1}')
print(f'  w2={w2}  dL/dw2={dw2}')
print(f'  Perdida inicial: {L_ini}')

Estado inicial:
  w1=0.5  dL/dw1=2.0
  w2=-0.3  dL/dw2=-1.5
  Perdida inicial: 4.2


In [11]:
# TODO: calcula los nuevos valores usando la regla w_nuevo = w - lr * dw
w1_nuevo = None  # reemplaza con tu calculo
w2_nuevo = None

print(f'w1 nuevo: {w1_nuevo}  (era {w1})')
print(f'w2 nuevo: {w2_nuevo}  (era {w2})')

w1 nuevo: None  (era 0.5)
w2 nuevo: None  (era -0.3)


**Pista:** `w1_nuevo = w1 - lr * dw1`.
Como `dw1 > 0`, el peso baja (se resta un número positivo).
Como `dw2 < 0`, el peso sube (se resta un número negativo).

In [12]:
# SOLUCION
w1_s = w1 - lr * dw1
w2_s = w2 - lr * dw2

print('=== Solucion (lr=0.1) ===')
print(f'w1 = {w1} - {lr}*{dw1} = {w1_s:.4f}  [bajo: grad positivo => bajar w reduce L]')
print(f'w2 = {w2} - {lr}*({dw2}) = {w2_s:.4f}  [subio: grad negativo => subir w reduce L]')
print()

lr_alto = 0.8
print(f'=== Con lr={lr_alto} ===')
w1_alt = w1 - lr_alto * dw1
w2_alt = w2 - lr_alto * dw2
print(f'w1 = {w1_alt:.4f}  (cambio: {abs(w1_alt-w1):.4f} vs {abs(w1_s-w1):.4f} con lr=0.1)')
print(f'w2 = {w2_alt:.4f}  (cambio: {abs(w2_alt-w2):.4f} vs {abs(w2_s-w2):.4f} con lr=0.1)')
print()
print('Con lr=0.8 los pasos son 8x mayores.')
print('Para redes con superficies de perdida no convexas, pasos tan grandes')
print('pueden saltarse el minimo o provocar divergencia (loss = NaN).')

=== Solucion (lr=0.1) ===
w1 = 0.5 - 0.1*2.0 = 0.3000  [bajo: grad positivo => bajar w reduce L]
w2 = -0.3 - 0.1*(-1.5) = -0.1500  [subio: grad negativo => subir w reduce L]

=== Con lr=0.8 ===
w1 = -1.1000  (cambio: 1.6000 vs 0.2000 con lr=0.1)
w2 = 0.9000  (cambio: 1.2000 vs 0.1500 con lr=0.1)

Con lr=0.8 los pasos son 8x mayores.
Para redes con superficies de perdida no convexas, pasos tan grandes
pueden saltarse el minimo o provocar divergencia (loss = NaN).


## 7. Ejercicio de decisión: cuando NO usar una red neuronal

**Caso A - la empresa Supply Chain:**
El equipo quiere predecir si un pedido llegará con retraso dado: proveedor,
fecha, artículo, destino y cantidad. Tienen 50.000 registros históricos con
la variable objetivo 'retraso (si/no)'. El modelo prioriza que pedidos vigilar.

**Caso B - la empresa Finanzas:**
El área quiere calcular el margen bruto esperado de un cliente.
El margen depende de precio de venta (fijo por contrato), volumen
(estimado por comercial) y coste de producto (conocido).
Fórmula: `margen = (precio - coste) * volumen`.

**Caso C - la empresa RRHH:**
El área quiere identificar empleados en riesgo de baja. Tienen 200 registros
de empleados con datos de encuestas, asistencia y rendimiento. La decisión
de intervención la toma el director de RRHH de forma individual y auditada.

**Pregunta:** Para cada caso, ¿usarías una red neuronal? ¿por que sí o no?
¿Que alternativa propones cuando decides no usarla?

**Tu razonamiento:**

(escribe aquí tu argumentación caso por caso)

---

**Criterios de evaluación (instructor):**

**Caso A:** Red neuronal o gradient boosting justificado. Dataset suficiente,
problema no lineal, interpretabilidad no crítica (solo prioriza supervisión humana).
Respuesta madura: proponer gradient boosting como baseline antes de escalar a red.

**Caso B:** NO se necesita ML. La fórmula es exacta y conocida. Usar una red
aquí sería un error de categoría: hay solución analítica directa.
El modelo 'aprendería' algo que ya se sabe calcular. Rechazar la propuesta.

**Caso C:** NO se recomienda red neuronal ni ML complejo. Dataset de 200 registros
es insuficiente. La decisión individual exige interpretabilidad auditada.
Proponer: regresión logística o árbol de decisión simple, o scoring por reglas.

## 8. Assets generados

| Fichero | Contenido |
|---|---|
| `images/B03E_fig01.png` | Función de pérdida 1D con gradiente en dos puntos |
| `images/B03E_fig02.png` | Descenso 2D con curvas de nivel y trayectoria |
| `images/B03E_fig03.png` | Comparativa de learning rates y curvas de pérdida |
| `images/B03E_fig04.html` | Superficie de pérdida 3D interactiva (Plotly) |
| `images/B03E_fig05.png` | Tabla comparativa mini-batch / SGD / full-batch |
| `images/B03E_fig06.png` | Esquema ciclo de entrenamiento |

**Dependencias:** numpy, pandas, matplotlib, plotly